In [1]:
"""Phase D Benchmark: compacted vs predicated vs dense attention.

Paste this ENTIRE cell into a Kaggle TPU notebook. Zero imports from the repo.
Uses standard JAX ops only — no Pallas, no BlockSpec, no Mosaic.

This measures the ACTUAL Δτ from stream compaction: fewer blocks gathered
and matmul'd means fewer FLOPs means less wall-clock time.
"""

import jax
import jax.numpy as jnp
import time

BS = 512; HD = 128; NH = 4; WARMUP = 10; REPS = 30

# ============================================================================
# Data generation
# ============================================================================
def make_data(sl):
    k1,k2,k3 = jax.random.split(jax.random.PRNGKey(0), 3)
    return (jax.random.normal(k1, (1,NH,HD), dtype=jnp.bfloat16),
            jax.random.normal(k2, (sl,NH,HD), dtype=jnp.bfloat16),
            jax.random.normal(k3, (sl,NH,HD), dtype=jnp.bfloat16))

def make_mask(nb, pct):
    if pct == 0: return jnp.ones((nb,NH), dtype=jnp.bool_)
    n = int(nb * pct / 100)
    m = jnp.ones(nb, dtype=jnp.bool_).at[:n].set(False)
    return jnp.broadcast_to(
        m[jax.random.permutation(jax.random.PRNGKey(42), nb)][:, None],
        (nb, NH))

def bench(label, fn):
    for _ in range(WARMUP): fn().block_until_ready()
    ts = []
    for _ in range(REPS):
        t0 = time.perf_counter(); fn().block_until_ready()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort(); med = ts[len(ts)//2]
    print(f"  {label}: {med:.3f} ms")
    return med

# ============================================================================
# Stream compaction (pure JAX — emulates the HLO pass prefix sum)
# ============================================================================
def stream_compact(mask):
    if mask.ndim == 2: mask_1d = jnp.any(mask, axis=1)
    else: mask_1d = mask
    nb = mask_1d.shape[0]
    iota = jnp.arange(nb, dtype=jnp.int32)
    keys = jnp.where(mask_1d, iota, nb + iota)
    return jnp.argsort(keys, stable=True), jnp.sum(mask_1d).astype(jnp.int32)

# ============================================================================
# Three attention implementations
# ============================================================================

@jax.jit
def dense_attn(q, k, v):
    """Baseline: full dense attention, no masking."""
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)

@jax.jit
def predicated_attn(q, k, v, mask):
    """v1: compute ALL blocks, mask logits. MXU still fires for evicted blocks."""
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    token_mask = jnp.repeat(mask, BS, axis=0)  # (seq_k, NH)
    lo = jnp.where(token_mask[None,:,:], lo, -1e9)
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)

@jax.jit
def compacted_attn(q, k, v, mask):
    """v2: stream compact → gather active blocks → dense attention on K blocks only."""
    nb = k.shape[0] // BS
    active_idx, num_active = stream_compact(mask)

    # Gather active blocks
    k_blocks = k.reshape(nb, BS, NH, HD)
    v_blocks = v.reshape(nb, BS, NH, HD)
    k_comp = k_blocks[active_idx]  # (nb, BS, NH, HD) — first K are valid
    v_comp = v_blocks[active_idx]
    k_flat = k_comp.reshape(nb * BS, NH, HD)
    v_flat = v_comp.reshape(nb * BS, NH, HD)

    # Validity mask: only first num_active * BS tokens are real
    valid = jnp.arange(nb * BS) < (num_active * BS)

    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k_flat.astype(jnp.float32)) / sc
    lo = jnp.where(valid[None,:,None], lo, -1e9)
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v_flat.astype(jnp.float32)).astype(jnp.bfloat16)

# ============================================================================
# Run benchmarks
# ============================================================================
print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}, warmup={WARMUP}, reps={REPS}")

results = {}
for seq in [8192, 16384, 32768, 65536]:
    nb = seq // BS
    q, k, v = make_data(seq)
    print(f"\n{'='*70}")
    print(f"  {seq} tokens ({nb} blocks)")
    print(f"{'='*70}")

    t_dense = bench("dense_0%", lambda: dense_attn(q, k, v))

    for pct in [0, 25, 50, 75, 90]:
        m = make_mask(nb, pct)
        t_pred = bench(f"predicated_{pct}%", lambda m=m: predicated_attn(q, k, v, m))
        t_comp = bench(f"compacted_{pct}%",  lambda m=m: compacted_attn(q, k, v, m))
        results[(seq, pct)] = {'dense': t_dense, 'pred': t_pred, 'comp': t_comp}

# Summary
print(f"\n{'='*70}")
print(f"  SUMMARY: Δτ = (predicated - compacted) / predicated")
print(f"{'='*70}")
print(f"{'seq':>6} | {'evict%':>6} | {'pred ms':>8} | {'comp ms':>8} | {'Δτ':>8} | {'speedup':>8}")
print("-" * 60)
for (seq, pct), r in sorted(results.items()):
    dt = (r['pred'] - r['comp']) / r['pred'] * 100
    spd = r['pred'] / r['comp'] if r['comp'] > 0 else 0
    print(f"{seq:>6} | {pct:>6} | {r['pred']:>8.3f} | {r['comp']:>8.3f} | {dt:>7.1f}% | {spd:>7.2f}x")


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1780418857.473981      14 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Platform: TPU v5 lite, JAX 0.10.1
Config: Q=1, heads=4, d=128, block=512, warmup=10, reps=30

  8192 tokens (16 blocks)
  dense_0%: 0.252 ms
  predicated_0%: 0.254 ms
  compacted_0%: 0.398 ms
  predicated_25%: 0.246 ms
  compacted_25%: 0.392 ms
  predicated_50%: 0.245 ms
  compacted_50%: 0.389 ms
  predicated_75%: 0.244 ms
  compacted_75%: 0.394 ms
  predicated_90%: 0.246 ms
  compacted_90%: 0.391 ms

  16384 tokens (32 blocks)
  dense_0%: 0.433 ms
  predicated_0%: 0.431 ms
  compacted_0%: 0.547 ms
  predicated_25%: 0.428 ms
  compacted_25%: 0.523 ms
  predicated_50%: 0.397 ms
  compacted_50%: 0.504 ms
  predicated_75%: 0.397 ms
  compacted_75%: 0.491 ms
  predicated_90%: 0.395 ms
  compacted_90%: 0.496 ms

  32768 tokens (64 blocks)
  dense_0%: 0.395 ms
  predicated_0%: 0.418 ms
  compacted_0%: 0.604 ms
  predicated_25%: 0.411 ms
  compacted_25%: 0.611 ms
  predicated_50%: 0.402 ms
  compacted_50%: 0.604 ms
  predicated_75%: 0.399 ms
  compacted_75%: 0.605 ms
  predicated_90%: 0.419 m

In [2]:
import jax, jax.numpy as jnp, time
from functools import partial

BS=512; HD=128; NH=4; WARMUP=10; REPS=30

def make_data(sl):
    k1,k2,k3=jax.random.split(jax.random.PRNGKey(0),3)
    return (jax.random.normal(k1,(1,NH,HD),dtype=jnp.bfloat16),
            jax.random.normal(k2,(sl,NH,HD),dtype=jnp.bfloat16),
            jax.random.normal(k3,(sl,NH,HD),dtype=jnp.bfloat16))

def make_mask(nb,pct):
    if pct==0: return jnp.ones((nb,NH),dtype=jnp.bool_)
    n=int(nb*pct/100); m=jnp.ones(nb,dtype=jnp.bool_).at[:n].set(False)
    return jnp.broadcast_to(m[jax.random.permutation(jax.random.PRNGKey(42),nb)][:,None],(nb,NH))

def bench(label,fn):
    for _ in range(WARMUP): fn().block_until_ready()
    ts=[]
    for _ in range(REPS):
        t0=time.perf_counter(); fn().block_until_ready(); ts.append((time.perf_counter()-t0)*1000)
    ts.sort(); med=ts[len(ts)//2]; print(f"  {label}: {med:.3f} ms"); return med

def stream_compact(mask):
    if mask.ndim==2: mask_1d=jnp.any(mask,axis=1)
    else: mask_1d=mask
    nb=mask_1d.shape[0]
    iota=jnp.arange(nb,dtype=jnp.int32)
    keys=jnp.where(mask_1d,iota,nb+iota)
    return jnp.argsort(keys,stable=True), jnp.sum(mask_1d).astype(jnp.int32)

# Predicated: full matmul, mask logits
@jax.jit
def predicated(q,k,v,mask):
    sc=jnp.sqrt(jnp.float32(HD))
    lo=jnp.einsum('qhd,khd->qkh',q.astype(jnp.float32),k.astype(jnp.float32))/sc
    tm=jnp.repeat(mask,BS,axis=0)
    lo=jnp.where(tm[None,:,:],lo,-1e9)
    w=jax.nn.softmax(lo,axis=1)
    return jnp.einsum('qkh,khd->qhd',w,v.astype(jnp.float32)).astype(jnp.bfloat16)

# Bucketed: gather K blocks → matmul on (bucket*BS) tokens, NOT (M*BS)
@partial(jax.jit, static_argnums=(4,))
def bucketed(q, k_blocks, v_blocks, active_idx, bucket):
    """Attention on exactly `bucket` blocks. Bucket is STATIC → smaller matmul."""
    idx = active_idx[:bucket]
    ka = k_blocks[idx].reshape(bucket*BS, NH, HD)  # SMALLER tensor
    va = v_blocks[idx].reshape(bucket*BS, NH, HD)
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), ka.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, va.astype(jnp.float32)).astype(jnp.bfloat16)

def next_bucket(k, max_blocks):
    """Round up to nearest power of 2, capped at max_blocks."""
    if k <= 0: return 1
    b = 1
    while b < k: b *= 2
    return min(b, max_blocks)

print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}")

for seq in [8192, 16384, 32768, 65536]:
    nb = seq // BS
    q, k, v = make_data(seq)
    k_blocks = k.reshape(nb, BS, NH, HD)
    v_blocks = v.reshape(nb, BS, NH, HD)

    print(f"\n{'='*70}")
    print(f"  {seq} tokens ({nb} blocks)")
    print(f"{'='*70}")

    for pct in [0, 50, 75, 90]:
        m = make_mask(nb, pct)
        t_pred = bench(f"pred_{pct}%", lambda m=m: predicated(q,k,v,m))

        aidx, nact = stream_compact(m)
        nact_val = int(nact)
        bkt = next_bucket(nact_val, nb)

        # Warmup the bucket (first call compiles)
        for _ in range(3): bucketed(q, k_blocks, v_blocks, aidx, bkt).block_until_ready()

        t_bkt = bench(f"bucket_{pct}% (K={nact_val},bkt={bkt})",
                      lambda ai=aidx, b=bkt: bucketed(q, k_blocks, v_blocks, ai, b))

        dt = (t_pred - t_bkt) / t_pred * 100
        print(f"    -> Δτ = {dt:+.1f}%  ({t_pred/t_bkt:.2f}x)")

Platform: TPU v5 lite, JAX 0.10.1
Config: Q=1, heads=4, d=128, block=512

  8192 tokens (16 blocks)
  pred_0%: 0.228 ms
  bucket_0% (K=16,bkt=16): 0.333 ms
    -> Δτ = -45.8%  (0.69x)
  pred_50%: 0.202 ms
  bucket_50% (K=8,bkt=8): 0.204 ms
    -> Δτ = -0.6%  (0.99x)
  pred_75%: 0.190 ms
  bucket_75% (K=4,bkt=4): 0.185 ms
    -> Δτ = +2.2%  (1.02x)
  pred_90%: 0.231 ms
  bucket_90% (K=2,bkt=2): 0.148 ms
    -> Δτ = +36.0%  (1.56x)

  16384 tokens (32 blocks)
  pred_0%: 0.383 ms
  bucket_0% (K=32,bkt=32): 0.484 ms
    -> Δτ = -26.4%  (0.79x)
  pred_50%: 0.390 ms
  bucket_50% (K=16,bkt=16): 0.345 ms
    -> Δτ = +11.4%  (1.13x)
  pred_75%: 0.383 ms
  bucket_75% (K=8,bkt=8): 0.196 ms
    -> Δτ = +48.8%  (1.95x)
  pred_90%: 0.391 ms
  bucket_90% (K=4,bkt=4): 0.166 ms
    -> Δτ = +57.4%  (2.35x)

  32768 tokens (64 blocks)
  pred_0%: 0.368 ms
  bucket_0% (K=64,bkt=64): 0.558 ms
    -> Δτ = -51.8%  (0.66x)
  pred_50%: 0.369 ms
  bucket_50% (K=32,bkt=32): 0.466 ms
    -> Δτ = -26.3%  (0.79x)
  

In [5]:
"""Phase D.5: XLA Loop Indirection Benchmark — fori_loop + dynamic_slice.
NO Pallas. NO intermediate buffer. NO libtpu version dependency.
DMA engine fetches directly from original KV-cache via computed address.
Loop runs exactly num_active iterations (dynamic bound, one JIT).
Paste this entire cell into a Kaggle TPU v5 notebook.
"""
import jax
import jax.numpy as jnp
from functools import partial
import time
BS = 512; HD = 128; NH = 4; WARMUP = 10; REPS = 30
# ============================================================
# Data & utilities (identical to previous benchmarks)
# ============================================================
def make_data(sl):
    k1, k2, k3 = jax.random.split(jax.random.PRNGKey(0), 3)
    return (
        jax.random.normal(k1, (1, NH, HD), dtype=jnp.bfloat16),
        jax.random.normal(k2, (sl, NH, HD), dtype=jnp.bfloat16),
        jax.random.normal(k3, (sl, NH, HD), dtype=jnp.bfloat16),
    )
def make_mask(nb, pct):
    if pct == 0: return jnp.ones(nb, dtype=jnp.bool_)
    n = int(nb * pct / 100)
    m = jnp.ones(nb, dtype=jnp.bool_).at[:n].set(False)
    return m[jax.random.permutation(jax.random.PRNGKey(42), nb)]
def stream_compact(mask):
    nb = mask.shape[0]
    iota = jnp.arange(nb, dtype=jnp.int32)
    keys = jnp.where(mask, iota, nb + iota)
    return jnp.argsort(keys, stable=True), jnp.sum(mask).astype(jnp.int32)
def bench(label, fn):
    for _ in range(WARMUP): fn().block_until_ready()
    ts = []
    for _ in range(REPS):
        t0 = time.perf_counter(); fn().block_until_ready()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort(); med = ts[len(ts)//2]
    print(f"  {label}: {med:.3f} ms")
    return med
# ============================================================
# XLA loop indirection: fori_loop + dynamic_slice
# ============================================================
@jax.jit
def indirect_loop_attn(q, k_cache, v_cache, active_indices, num_active):
    """OrthoCache attention via native XLA loop indirection.
    - fori_loop compiles to a hardware loop on TPU
    - dynamic_slice compiles to DMA with computed address
    - No intermediate buffer, no gather, no Pallas
    - num_active is a TRACED int — one JIT handles all eviction rates
    """
    seq_q, num_heads, head_dim = q.shape
    scale = jnp.sqrt(jnp.float32(head_dim))
    # Online softmax accumulators
    m_init = jnp.full((seq_q, num_heads), -1e30, dtype=jnp.float32)
    l_init = jnp.zeros((seq_q, num_heads), dtype=jnp.float32)
    o_init = jnp.zeros((seq_q, num_heads, head_dim), dtype=jnp.float32)
    def body(i, carry):
        m_prev, l_prev, o_prev = carry
        # Scalar register read: which block to fetch
        real_idx = active_indices[i]
        # ONE HBM TOUCH: DMA fetches directly from original cache
        k_block = jax.lax.dynamic_slice(
            k_cache,
            (real_idx * BS, 0, 0),
            (BS, num_heads, head_dim)
        )
        v_block = jax.lax.dynamic_slice(
            v_cache,
            (real_idx * BS, 0, 0),
            (BS, num_heads, head_dim)
        )
        # Logits: (seq_q, BS, NH)
        logits = jnp.einsum('qhd,khd->qkh',
                            q.astype(jnp.float32),
                            k_block.astype(jnp.float32)) / scale
        # Online softmax update
        m_block = jnp.max(logits, axis=1)              # (seq_q, NH)
        m_next = jnp.maximum(m_prev, m_block)
        exp_logits = jnp.exp(logits - m_next[:, None, :])
        exp_prev = jnp.exp(m_prev - m_next)
        l_block = jnp.sum(exp_logits, axis=1)           # (seq_q, NH)
        l_next = l_prev * exp_prev + l_block
        # Weighted V: (seq_q, NH, HD)
        v_agg = jnp.einsum('qkh,khd->qhd',
                           exp_logits,
                           v_block.astype(jnp.float32))
        o_next = o_prev * exp_prev[:, :, None] + v_agg
        return m_next, l_next, o_next
    m_f, l_f, o_f = jax.lax.fori_loop(0, num_active, body, (m_init, l_init, o_init))
    return (o_f / l_f[:, :, None]).astype(jnp.bfloat16)
# ============================================================
# Baselines (identical to previous benchmarks)
# ============================================================
@jax.jit
def dense_attn(q, k, v):
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)
@jax.jit
def predicated_attn(q, k, v, mask):
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    lo = jnp.where(jnp.repeat(mask, BS)[None, :, None], lo, -1e9)
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)
# ============================================================
# Correctness check
# ============================================================
print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}")
print(f"Method: jax.lax.fori_loop + jax.lax.dynamic_slice (native XLA)\n")
print("=== Correctness check (16K, 50% eviction) ===")
q_c, k_c, v_c = make_data(16384)
m_c = make_mask(32, 50)
aidx_c, nact_c = stream_compact(m_c)
out_pred  = predicated_attn(q_c, k_c, v_c, m_c)
out_loop  = indirect_loop_attn(q_c, k_c, v_c, aidx_c, nact_c)
err = jnp.max(jnp.abs(out_pred.astype(jnp.float32) - out_loop.astype(jnp.float32)))
print(f"  predicated vs loop-indirect max err: {err:.6f}")
assert err < 0.05, f"Output mismatch: {err}"
print("  PASSED\n")
# ============================================================
# Benchmark
# ============================================================
results = {}
for seq in [8192, 16384, 32768, 65536]:
    nb = seq // BS
    q, k, v = make_data(seq)
    print(f"{'='*70}")
    print(f"  {seq} tokens ({nb} blocks)")
    print(f"{'='*70}")
    t_dense = bench("dense", lambda: dense_attn(q, k, v))
    for pct in [0, 50, 75, 90]:
        m = make_mask(nb, pct)
        t_pred = bench(f"pred_{pct}%", lambda m=m: predicated_attn(q, k, v, m))
        aidx, nact = stream_compact(m)
        t_loop = bench(f"loop_{pct}% (K={int(nact)})",
                      lambda ai=aidx, na=nact: indirect_loop_attn(q, k, v, ai, na))
        dt = (t_pred - t_loop) / t_pred * 100
        spd = t_pred / t_loop if t_loop > 0 else 0
        print(f"    -> Δτ = {dt:+.1f}%  ({spd:.2f}x)")
        results[(seq, pct)] = {
            'dense': t_dense, 'pred': t_pred,
            'loop': t_loop, 'K': int(nact)
        }
# ============================================================
# Summary
# ============================================================
print(f"\n{'='*70}")
print(f"  SUMMARY: XLA Loop Indirection vs Predicated")
print(f"  Key: 0% eviction should be ~1.00x (not 0.33x)")
print(f"{'='*70}")
print(f"{'seq':>6} | {'evict%':>6} | {'K':>4} | {'pred ms':>8} | {'loop ms':>8} | {'Δτ':>8} | {'speedup':>8}")
print("-" * 68)
for (seq, pct), r in sorted(results.items()):
    dt = (r['pred'] - r['loop']) / r['pred'] * 100
    spd = r['pred'] / r['loop'] if r['loop'] > 0 else 0
    print(f"{seq:>6} | {pct:>6} | {r['K']:>4} | {r['pred']:>8.3f} | {r['loop']:>8.3f} | {dt:>7.1f}% | {spd:>7.2f}x")

Platform: TPU v5 lite, JAX 0.10.1
Config: Q=1, heads=4, d=128, block=512
Method: jax.lax.fori_loop + jax.lax.dynamic_slice (native XLA)

=== Correctness check (16K, 50% eviction) ===
  predicated vs loop-indirect max err: 0.000000
  PASSED

  8192 tokens (16 blocks)
  dense: 0.264 ms
  pred_0%: 0.257 ms
  loop_0% (K=16): 0.334 ms
    -> Δτ = -30.2%  (0.77x)
  pred_50%: 0.271 ms
  loop_50% (K=8): 0.260 ms
    -> Δτ = +4.1%  (1.04x)
  pred_75%: 0.284 ms
  loop_75% (K=4): 0.223 ms
    -> Δτ = +21.7%  (1.28x)
  pred_90%: 0.256 ms
  loop_90% (K=2): 0.191 ms
    -> Δτ = +25.6%  (1.34x)
  16384 tokens (32 blocks)
  dense: 0.418 ms
  pred_0%: 0.428 ms
  loop_0% (K=32): 0.450 ms
    -> Δτ = -5.2%  (0.95x)
  pred_50%: 0.427 ms
  loop_50% (K=16): 0.346 ms
    -> Δτ = +19.0%  (1.23x)
  pred_75%: 0.431 ms
  loop_75% (K=8): 0.254 ms
    -> Δτ = +41.1%  (1.70x)
  pred_90%: 0.458 ms
  loop_90% (K=4): 0.233 ms
    -> Δτ = +49.1%  (1.96x)
  32768 tokens (64 blocks)
  dense: 0.446 ms
  pred_0%: 0.439 ms


In [1]:
"""Phase D.7: Vectorized Indirect Attention — vmap + fori_loop + dynamic_slice.
Tests whether vmap over heads erases the 0% eviction tax by:
1. Merging per-head DMA descriptors into wide vector-aligned bursts
2. Packing the MXU operands across heads for optimal systolic array density
3. Hiding per-iteration DMA latency behind cross-head parallelism
Compares:
  - D.5 baseline: fori_loop with multi-head einsum inside loop body
  - D.7 vmap-heads: single-head fori_loop vmapped over NH heads
  - D.7 vmap-batch: batch + heads vmapped (B=4)
Paste this entire cell into a Kaggle TPU v5 notebook.
"""
import jax
import jax.numpy as jnp
from functools import partial
import time
BS = 512; HD = 128; NH = 4; WARMUP = 10; REPS = 30
# ============================================================
# Data & utilities
# ============================================================
def make_data(sl, batch=1):
    k1, k2, k3 = jax.random.split(jax.random.PRNGKey(0), 3)
    return (
        jax.random.normal(k1, (batch, 1, NH, HD), dtype=jnp.bfloat16),
        jax.random.normal(k2, (batch, sl, NH, HD), dtype=jnp.bfloat16),
        jax.random.normal(k3, (batch, sl, NH, HD), dtype=jnp.bfloat16),
    )
def make_mask(nb, pct):
    if pct == 0: return jnp.ones(nb, dtype=jnp.bool_)
    n = int(nb * pct / 100)
    m = jnp.ones(nb, dtype=jnp.bool_).at[:n].set(False)
    return m[jax.random.permutation(jax.random.PRNGKey(42), nb)]
def stream_compact(mask):
    nb = mask.shape[0]
    iota = jnp.arange(nb, dtype=jnp.int32)
    keys = jnp.where(mask, iota, nb + iota)
    return jnp.argsort(keys, stable=True), int(jnp.sum(mask))
def bench(label, fn):
    for _ in range(WARMUP): fn().block_until_ready()
    ts = []
    for _ in range(REPS):
        t0 = time.perf_counter(); fn().block_until_ready()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort(); med = ts[len(ts)//2]
    print(f"  {label}: {med:.3f} ms")
    return med
# ============================================================
# CORE: Single-head indirection kernel
# ============================================================
def single_head_indirect_attn(q, k_cache, v_cache, active_indices, num_active):
    """Single head, single batch element.
    q: (1, HD), k_cache: (SL, HD), v_cache: (SL, HD)
    active_indices: (M,) int32, num_active: scalar int
    """
    seq_q, head_dim = q.shape
    scale = jnp.sqrt(jnp.float32(head_dim))
    m_init = jnp.full((seq_q,), -1e30, dtype=jnp.float32)
    l_init = jnp.zeros((seq_q,), dtype=jnp.float32)
    o_init = jnp.zeros((seq_q, head_dim), dtype=jnp.float32)
    def body(i, carry):
        m_prev, l_prev, o_prev = carry
        real_idx = active_indices[i]
        k_block = jax.lax.dynamic_slice(k_cache, (real_idx * BS, 0), (BS, head_dim))
        v_block = jax.lax.dynamic_slice(v_cache, (real_idx * BS, 0), (BS, head_dim))
        logits = jnp.einsum('qd,kd->qk',
                            q.astype(jnp.float32),
                            k_block.astype(jnp.float32)) / scale
        m_block = jnp.max(logits, axis=1)
        m_next = jnp.maximum(m_prev, m_block)
        exp_logits = jnp.exp(logits - m_next[:, None])
        exp_prev = jnp.exp(m_prev - m_next)
        l_block = jnp.sum(exp_logits, axis=1)
        l_next = l_prev * exp_prev + l_block
        v_agg = jnp.einsum('qk,kd->qd', exp_logits, v_block.astype(jnp.float32))
        o_next = o_prev * exp_prev[:, None] + v_agg
        return m_next, l_next, o_next
    m_f, l_f, o_f = jax.lax.fori_loop(0, num_active, body, (m_init, l_init, o_init))
    return (o_f / l_f[:, None]).astype(jnp.bfloat16)
# ============================================================
# LAYER 1: vmap over heads (shared mask)
# ============================================================
# q: (1, NH, HD), k: (SL, NH, HD), v: (SL, NH, HD)
# active_indices: (M,) shared, num_active: scalar shared
_multi_head_indirect = jax.vmap(
    single_head_indirect_attn,
    in_axes=(1, 1, 1, None, None),
    out_axes=1
)
# ============================================================
# LAYER 2: vmap over batch (per-element mask)
# ============================================================
# q: (B, 1, NH, HD), k: (B, SL, NH, HD), v: (B, SL, NH, HD)
# active_indices: (B, M), num_active: scalar (same for all — bench uses uniform eviction)
_batch_multi_head_indirect = jax.vmap(
    _multi_head_indirect,
    in_axes=(0, 0, 0, 0, None),
    out_axes=0
)
# ============================================================
# JIT wrappers (num_active is static for optimal loop compilation)
# ============================================================
@partial(jax.jit, static_argnums=(4,))
def vmap_heads_attn(q, k, v, active_indices, num_active):
    """B=1: q (1, NH, HD), k (SL, NH, HD), v (SL, NH, HD)"""
    return _multi_head_indirect(q, k, v, active_indices, num_active)
@partial(jax.jit, static_argnums=(4,))
def vmap_batch_attn(q, k, v, active_indices, num_active):
    """B>1: q (B, 1, NH, HD), k (B, SL, NH, HD), v (B, SL, NH, HD)"""
    return _batch_multi_head_indirect(q, k, v, active_indices, num_active)
# ============================================================
# D.5 baseline (for comparison — multi-head einsum inside loop)
# ============================================================
@jax.jit
def d5_loop_attn(q, k, v, active_indices, num_active):
    """Phase D.5: fori_loop with multi-head einsum inside."""
    seq_q, num_heads, head_dim = q.shape
    scale = jnp.sqrt(jnp.float32(head_dim))
    m_init = jnp.full((seq_q, num_heads), -1e30, dtype=jnp.float32)
    l_init = jnp.zeros((seq_q, num_heads), dtype=jnp.float32)
    o_init = jnp.zeros((seq_q, num_heads, head_dim), dtype=jnp.float32)
    def body(i, carry):
        m_prev, l_prev, o_prev = carry
        real_idx = active_indices[i]
        k_block = jax.lax.dynamic_slice(k, (real_idx * BS, 0, 0), (BS, num_heads, head_dim))
        v_block = jax.lax.dynamic_slice(v, (real_idx * BS, 0, 0), (BS, num_heads, head_dim))
        logits = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k_block.astype(jnp.float32)) / scale
        m_block = jnp.max(logits, axis=1)
        m_next = jnp.maximum(m_prev, m_block)
        exp_logits = jnp.exp(logits - m_next[:, None, :])
        exp_prev = jnp.exp(m_prev - m_next)
        l_block = jnp.sum(exp_logits, axis=1)
        l_next = l_prev * exp_prev + l_block
        v_agg = jnp.einsum('qkh,khd->qhd', exp_logits, v_block.astype(jnp.float32))
        o_next = o_prev * exp_prev[:, :, None] + v_agg
        return m_next, l_next, o_next
    m_f, l_f, o_f = jax.lax.fori_loop(0, num_active, body, (m_init, l_init, o_init))
    return (o_f / l_f[:, :, None]).astype(jnp.bfloat16)
# Dense baseline
@jax.jit
def dense_attn(q, k, v):
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)
@jax.jit
def predicated_attn(q, k, v, mask):
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    lo = jnp.where(jnp.repeat(mask, BS)[None, :, None], lo, -1e9)
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)
# ============================================================
# Correctness check
# ============================================================
print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}")
print(f"Method: vmap(single_head_fori_loop) over heads + batch\n")
print("=== Correctness check (16K, 50% eviction) ===")
q_c1, k_c1, v_c1 = make_data(16384, batch=1)
q_1 = q_c1[0]; k_1 = k_c1[0]; v_1 = v_c1[0]  # squeeze batch dim
m_c = make_mask(32, 50)
aidx_c, nact_c = stream_compact(m_c)
out_d5   = d5_loop_attn(q_1, k_1, v_1, aidx_c, nact_c)
out_vmap = vmap_heads_attn(q_1, k_1, v_1, aidx_c, nact_c)
err = jnp.max(jnp.abs(out_d5.astype(jnp.float32) - out_vmap.astype(jnp.float32)))
print(f"  D.5 loop vs vmap-heads max err: {err:.6f}")
assert err < 0.01, f"Output mismatch: {err}"
print("  PASSED\n")
# Batch correctness
print("=== Batch correctness (B=4, 16K, 50%) ===")
q_b4, k_b4, v_b4 = make_data(16384, batch=4)
aidx_b4 = jnp.broadcast_to(aidx_c[None, :], (4, aidx_c.shape[0]))
out_batch = vmap_batch_attn(q_b4, k_b4, v_b4, aidx_b4, nact_c)
out_single = vmap_heads_attn(q_b4[0], k_b4[0], v_b4[0], aidx_c, nact_c)
err_b = jnp.max(jnp.abs(out_batch[0].astype(jnp.float32) - out_single.astype(jnp.float32)))
print(f"  batch[0] vs single max err: {err_b:.6f}")
assert err_b < 0.01, f"Batch mismatch: {err_b}"
print("  PASSED\n")
# ============================================================
# Benchmark: B=1 comparison (D.5 vs vmap-heads vs dense)
# ============================================================
print("=" * 70)
print("  SECTION 1: B=1 — D.5 loop vs vmap-heads vs dense/predicated")
print("=" * 70)
results_b1 = {}
for seq in [8192, 16384, 32768, 65536]:
    nb = seq // BS
    q_b, k_b, v_b = make_data(seq, batch=1)
    q_1 = q_b[0]; k_1 = k_b[0]; v_1 = v_b[0]
    print(f"\n{'='*70}")
    print(f"  {seq} tokens ({nb} blocks)")
    print(f"{'='*70}")
    t_dense = bench("dense", lambda: dense_attn(q_1, k_1, v_1))
    for pct in [0, 50, 75, 90]:
        m = make_mask(nb, pct)
        aidx, nact = stream_compact(m)
        t_pred = bench(f"pred_{pct}%",
                      lambda m=m: predicated_attn(q_1, k_1, v_1, m))
        # Warmup D.5
        for _ in range(3): d5_loop_attn(q_1, k_1, v_1, aidx, nact).block_until_ready()
        t_d5 = bench(f"d5_loop_{pct}% (K={nact})",
                    lambda ai=aidx, n=nact: d5_loop_attn(q_1, k_1, v_1, ai, n))
        # Warmup vmap-heads
        for _ in range(3): vmap_heads_attn(q_1, k_1, v_1, aidx, nact).block_until_ready()
        t_vh = bench(f"vmap_heads_{pct}% (K={nact})",
                    lambda ai=aidx, n=nact: vmap_heads_attn(q_1, k_1, v_1, ai, n))
        dt_d5 = (t_pred - t_d5) / t_pred * 100
        dt_vh = (t_pred - t_vh) / t_pred * 100
        print(f"    D.5:  Δτ = {dt_d5:+.1f}%  ({t_pred/t_d5:.2f}x)")
        print(f"    vmap: Δτ = {dt_vh:+.1f}%  ({t_pred/t_vh:.2f}x)")
        results_b1[(seq, pct)] = {
            'dense': t_dense, 'pred': t_pred,
            'd5': t_d5, 'vmap': t_vh, 'K': nact
        }
# ============================================================
# Benchmark: B=4 batched
# ============================================================
print(f"\n{'='*70}")
print("  SECTION 2: B=4 — vmap-batch")
print(f"{'='*70}")
results_b4 = {}
for seq in [8192, 16384, 32768, 65536]:
    nb = seq // BS
    q_b4, k_b4, v_b4 = make_data(seq, batch=4)
    print(f"\n  {seq} tokens ({nb} blocks), B=4")
    for pct in [0, 50, 75, 90]:
        m = make_mask(nb, pct)
        aidx, nact = stream_compact(m)
        aidx_batched = jnp.broadcast_to(aidx[None, :], (4, aidx.shape[0]))
        for _ in range(3):
            vmap_batch_attn(q_b4, k_b4, v_b4, aidx_batched, nact).block_until_ready()
        t_vb = bench(f"  vmap_batch_{pct}% (K={nact})",
                    lambda ai=aidx_batched, n=nact: vmap_batch_attn(q_b4, k_b4, v_b4, ai, n))
        results_b4[(seq, pct)] = {'vmap_batch': t_vb, 'K': nact}
# ============================================================
# Summary
# ============================================================
print(f"\n{'='*70}")
print("  SUMMARY: B=1 — D.5 loop vs vmap-heads vs predicated")
print(f"{'='*70}")
print(f"{'seq':>6} | {'ev%':>4} | {'K':>4} | {'pred':>7} | {'D.5':>7} | {'vmap':>7} | {'D.5 Δτ':>7} | {'vmap Δτ':>7}")
print("-" * 72)
for (seq, pct), r in sorted(results_b1.items()):
    dt_d5 = (r['pred'] - r['d5']) / r['pred'] * 100
    dt_vh = (r['pred'] - r['vmap']) / r['pred'] * 100
    print(f"{seq:>6} | {pct:>4} | {r['K']:>4} | {r['pred']:>6.3f} | {r['d5']:>6.3f} | {r['vmap']:>6.3f} | {dt_d5:>6.1f}% | {dt_vh:>6.1f}%")
print(f"\n{'='*70}")
print("  SUMMARY: B=4 — vmap-batch (per-element ms)")
print(f"{'='*70}")
print(f"{'seq':>6} | {'ev%':>4} | {'K':>4} | {'total ms':>9} | {'per-elem':>9}")
print("-" * 50)
for (seq, pct), r in sorted(results_b4.items()):
    total = r['vmap_batch']
    per_elem = total / 4
    print(f"{seq:>6} | {pct:>4} | {r['K']:>4} | {total:>8.3f} | {per_elem:>8.3f}")

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1780434767.955619      14 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Platform: TPU v5 lite, JAX 0.10.1
Config: Q=1, heads=4, d=128, block=512
Method: vmap(single_head_fori_loop) over heads + batch

=== Correctness check (16K, 50% eviction) ===
  D.5 loop vs vmap-heads max err: 0.000000
  PASSED

=== Batch correctness (B=4, 16K, 50%) ===
  batch[0] vs single max err: 0.000000
  PASSED

  SECTION 1: B=1 — D.5 loop vs vmap-heads vs dense/predicated

  8192 tokens (16 blocks)
  dense: 0.268 ms
  pred_0%: 0.263 ms
  d5_loop_0% (K=16): 0.420 ms
  vmap_heads_0% (K=16): 0.319 ms
    D.5:  Δτ = -59.5%  (0.63x)
    vmap: Δτ = -21.1%  (0.83x)
  pred_50%: 0.260 ms
  d5_loop_50% (K=8): 0.374 ms
  vmap_heads_50% (K=8): 0.264 ms
    D.5:  Δτ = -43.7%  (0.70x)
    vmap: Δτ = -1.5%  (0.99x)
  pred_75%: 0.274 ms
  d5_loop_75% (K=4): 0.352 ms
  vmap_heads_75% (K=4): 0.247 ms
    D.5:  Δτ = -28.6%  (0.78x)
    vmap: Δτ = +9.8%  (1.11x)
  pred_90%: 0.262 ms
  d5_loop_90% (K=2): 0.341 ms
  vmap_heads_90% (K=2): 0.249 ms
    D.5:  Δτ = -30.0%  (0.77x)
    vmap: Δτ = +4.9%  (1

In [2]:
"""Phase E: Multi-Host AllToAllv Benchmark — ICI Bandwidth Reduction.
Simulates P=8 sequence-parallel devices on Kaggle TPU v5 lite (8 cores).
Each device owns seq_len/P tokens of the KV-cache. The AllToAllv protocol
compacts locally, syncs counts, exchanges only active blocks, then runs
indirect attention on the received blocks.
Measures:
  1. Correctness: sharded output matches single-device dense attention
  2. Data volume: bytes per device (compacted vs dense)
  3. Latency: sharded OrthoCache vs sharded dense
  4. Scaling: 0% / 50% / 75% / 90% eviction (uniform + skewed masks)
Paste this entire cell into a Kaggle TPU v5 notebook.
"""
import jax
import jax.numpy as jnp
from jax import lax
from functools import partial
import time
import json
BS = 512; HD = 128; NH = 4; WARMUP = 5; REPS = 20
print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
P = jax.device_count()
print(f"Devices: {P} ({', '.join(str(d) for d in jax.devices())})")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}, devices={P}")
print(f"Method: pmap AllToAllv (Strategy C: static-buffer exchange)\n")
assert P >= 2, f"Need ≥2 devices for multi-device test, got {P}"
# ============================================================
# Utilities
# ============================================================
def make_mask(nb, pct):
    """Create block mask with pct% eviction."""
    if pct == 0: return jnp.ones(nb, dtype=jnp.bool_)
    n = max(1, int(nb * pct / 100))
    n = min(n, nb - 1)  # keep at least 1 block
    m = jnp.ones(nb, dtype=jnp.bool_).at[:n].set(False)
    return m[jax.random.permutation(jax.random.PRNGKey(42), nb)]
def stream_compact(mask):
    nb = mask.shape[0]
    iota = jnp.arange(nb, dtype=jnp.int32)
    keys = jnp.where(mask, iota, nb + iota)
    return jnp.argsort(keys, stable=True), jnp.sum(mask).astype(jnp.int32)
def bench(label, fn):
    for _ in range(WARMUP): fn().block_until_ready()
    ts = []
    for _ in range(REPS):
        t0 = time.perf_counter(); fn().block_until_ready()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort(); med = ts[len(ts)//2]
    print(f"  {label}: {med:.3f} ms")
    return med
# ============================================================
# Single-device dense attention (ground truth)
# ============================================================
@jax.jit
def dense_attn_single(q, k, v):
    """q: (1, NH, HD), k/v: (SL, NH, HD)"""
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)
# ============================================================
# Sharded dense attention (baseline — full AllToAll, no eviction)
# ============================================================
@partial(jax.pmap, axis_name='d', in_axes=(0, 0, 0), out_axes=0)
def sharded_dense_attn(q_shard, k_shard, v_shard):
    """Each device has SL/P tokens. AllGather full KV, then dense attention."""
    # AllGather: each device gets full KV cache
    k_full = lax.all_gather(k_shard, axis_name='d', axis=0, tiled=True)
    v_full = lax.all_gather(v_shard, axis_name='d', axis=0, tiled=True)
    # Dense attention on full cache
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh',
                    q_shard.astype(jnp.float32),
                    k_full.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    out = jnp.einsum('qkh,khd->qhd', w, v_full.astype(jnp.float32))
    return out.astype(jnp.bfloat16)
# ============================================================
# Sharded OrthoCache attention (AllToAllv Strategy C)
# ============================================================
def _pack_active(kv_shard, active_indices, num_active, max_blocks):
    """Pack active blocks into static buffer. kv_shard: (local_sl, NH, HD)."""
    local_sl = kv_shard.shape[0]
    nh = kv_shard.shape[1]
    hd = kv_shard.shape[2]
    buf = jnp.zeros((max_blocks * BS, nh, hd), dtype=kv_shard.dtype)
    def body(i, buf):
        idx = active_indices[i]
        block = lax.dynamic_slice(kv_shard, (idx * BS, 0, 0), (BS, nh, hd))
        buf = lax.dynamic_update_slice(buf, block, (i * BS, 0, 0))
        return buf
    return lax.fori_loop(0, num_active, body, buf)
def _indirect_attn_on_received(q, k_received, v_received, total_active):
    """Online softmax attention over total_active blocks from received KV."""
    seq_q, nh, hd = q.shape
    scale = jnp.sqrt(jnp.float32(hd))
    m_init = jnp.full((seq_q, nh), -1e30, dtype=jnp.float32)
    l_init = jnp.zeros((seq_q, nh), dtype=jnp.float32)
    o_init = jnp.zeros((seq_q, nh, hd), dtype=jnp.float32)
    def body(i, carry):
        m_prev, l_prev, o_prev = carry
        k_blk = lax.dynamic_slice(k_received, (i * BS, 0, 0), (BS, nh, hd))
        v_blk = lax.dynamic_slice(v_received, (i * BS, 0, 0), (BS, nh, hd))
        logits = jnp.einsum('qhd,khd->qkh',
                            q.astype(jnp.float32),
                            k_blk.astype(jnp.float32)) / scale
        m_blk = jnp.max(logits, axis=1)
        m_new = jnp.maximum(m_prev, m_blk)
        exp_l = jnp.exp(logits - m_new[:, None, :])
        exp_p = jnp.exp(m_prev - m_new)
        l_new = l_prev * exp_p + jnp.sum(exp_l, axis=1)
        o_new = (o_prev * exp_p[:, :, None] +
                 jnp.einsum('qkh,khd->qhd', exp_l, v_blk.astype(jnp.float32)))
        return m_new, l_new, o_new
    m_f, l_f, o_f = lax.fori_loop(0, total_active, body,
                                    (m_init, l_init, o_init))
    return (o_f / l_f[:, :, None]).astype(jnp.bfloat16)
@partial(jax.pmap, axis_name='d',
         in_axes=(0, 0, 0, 0),
         out_axes=0)
def sharded_orthocache_attn(q_shard, k_shard, v_shard, block_mask):
    """Sharded OrthoCache attention with AllToAllv (Strategy C).
    Each device:
      1. Compacts local KV (keeps only active blocks)
      2. AllGathers active counts
      3. AllGathers compacted KV buffers
      4. Runs indirect attention on received blocks
    After AllGather, each device sees P chunks of num_blocks each.
    Chunk d has active blocks packed at positions [0..count_d) and
    zeros at [count_d..num_blocks). We iterate over all P*num_blocks
    block slots but use predication to skip zero-padded slots.
    """
    local_sl = k_shard.shape[0]
    num_blocks = local_sl // BS
    # Step 1: Local compaction
    active_indices, num_active = stream_compact(block_mask)
    # Step 2: Pack into static buffer (size = num_blocks * BS)
    k_packed = _pack_active(k_shard, active_indices, num_active, num_blocks)
    v_packed = _pack_active(v_shard, active_indices, num_active, num_blocks)
    # Step 3: AllGather counts (tiny payload — P integers)
    # num_active is a scalar (0D) — reshape to (1,) for all_gather
    counts_1d = num_active.reshape(1)
    all_counts_1d = lax.all_gather(counts_1d, axis_name='d', axis=0, tiled=True)
    # all_counts_1d shape: (P,)
    total_active = jnp.sum(all_counts_1d)
    num_devices = all_counts_1d.shape[0]  # P, derived from all_gather output
    # Step 4: AllGather compacted KV buffers
    # Each device sends its packed buffer (static size num_blocks * BS)
    # Receiver gets P copies concatenated along axis 0
    k_all = lax.all_gather(k_packed, axis_name='d', axis=0, tiled=True)
    v_all = lax.all_gather(v_packed, axis_name='d', axis=0, tiled=True)
    # k_all shape: (P * num_blocks * BS, NH, HD)
    # Step 5: Indirect attention over ALL received blocks
    # Layout in k_all: P chunks of num_blocks blocks each.
    # In chunk d, blocks [0..count_d) are valid, [count_d..num_blocks) are zero-padded.
    # Build a validity mask for all P*num_blocks block positions.
    total_blocks = num_devices * num_blocks  # static
    block_iota = jnp.arange(total_blocks, dtype=jnp.int32)
    # For block position b, it belongs to device d = b // num_blocks
    # Its local index within device d is j = b % num_blocks
    # It's valid iff j < all_counts_1d[d]
    device_ids = block_iota // num_blocks
    local_ids = block_iota % num_blocks
    per_device_counts = all_counts_1d[device_ids]  # count for each block's device
    valid_mask = local_ids < per_device_counts  # (total_blocks,) bool
    # Sort to push valid blocks to front (stable: preserves order)
    sort_keys = jnp.where(valid_mask, block_iota, total_blocks + block_iota)
    sorted_indices = jnp.argsort(sort_keys, stable=True)
    # Run attention using sorted_indices[0:total_active]
    out = _indirect_attn_global(q_shard, k_all, v_all, sorted_indices, total_active)
    return out
def _indirect_attn_global(q, k_all, v_all, global_indices, total_active):
    """Attention over globally indexed blocks."""
    seq_q, nh, hd = q.shape
    scale = jnp.sqrt(jnp.float32(hd))
    m_init = jnp.full((seq_q, nh), -1e30, dtype=jnp.float32)
    l_init = jnp.zeros((seq_q, nh), dtype=jnp.float32)
    o_init = jnp.zeros((seq_q, nh, hd), dtype=jnp.float32)
    def body(i, carry):
        m_prev, l_prev, o_prev = carry
        block_idx = global_indices[i]
        k_blk = lax.dynamic_slice(k_all, (block_idx * BS, 0, 0), (BS, nh, hd))
        v_blk = lax.dynamic_slice(v_all, (block_idx * BS, 0, 0), (BS, nh, hd))
        logits = jnp.einsum('qhd,khd->qkh',
                            q.astype(jnp.float32),
                            k_blk.astype(jnp.float32)) / scale
        m_blk = jnp.max(logits, axis=1)
        m_new = jnp.maximum(m_prev, m_blk)
        exp_l = jnp.exp(logits - m_new[:, None, :])
        exp_p = jnp.exp(m_prev - m_new)
        l_new = l_prev * exp_p + jnp.sum(exp_l, axis=1)
        o_new = (o_prev * exp_p[:, :, None] +
                 jnp.einsum('qkh,khd->qhd', exp_l, v_blk.astype(jnp.float32)))
        return m_new, l_new, o_new
    m_f, l_f, o_f = lax.fori_loop(0, total_active, body,
                                    (m_init, l_init, o_init))
    return (o_f / l_f[:, :, None]).astype(jnp.bfloat16)
# ============================================================
# Data generation
# ============================================================
def make_sharded_data(seq_len):
    """Generate q, k, v and shard across P devices."""
    k1, k2, k3 = jax.random.split(jax.random.PRNGKey(0), 3)
    q = jax.random.normal(k1, (1, NH, HD), dtype=jnp.bfloat16)
    k = jax.random.normal(k2, (seq_len, NH, HD), dtype=jnp.bfloat16)
    v = jax.random.normal(k3, (seq_len, NH, HD), dtype=jnp.bfloat16)
    # Shard: each device gets seq_len/P tokens
    local_sl = seq_len // P
    q_sharded = jnp.broadcast_to(q, (P, 1, NH, HD))  # replicate q
    k_sharded = k.reshape(P, local_sl, NH, HD)
    v_sharded = v.reshape(P, local_sl, NH, HD)
    return q, k, v, q_sharded, k_sharded, v_sharded
def make_uniform_masks(num_blocks_per_device, pct):
    """Same eviction rate on all devices."""
    masks = []
    for d in range(P):
        masks.append(make_mask(num_blocks_per_device, pct))
    return jnp.stack(masks)  # (P, num_blocks_per_device)
def make_skewed_masks(num_blocks_per_device):
    """Different eviction rates across devices (ascending retention)."""
    # Spread rates across however many devices we have
    all_rates = [90, 80, 75, 60, 50, 30, 20, 10]
    rates = all_rates[:P]
    masks = []
    for d in range(P):
        masks.append(make_mask(num_blocks_per_device, rates[d]))
    return jnp.stack(masks), rates
# ============================================================
# ICI data volume accounting
# ============================================================
def ici_accounting(seq_len, masks_per_device):
    """Compute ICI bytes transferred."""
    local_sl = seq_len // P
    local_blocks = local_sl // BS
    bytes_per_block = BS * NH * HD * 2  # bf16
    dense_bytes_per_device = local_blocks * bytes_per_block  # what each device would send dense
    total_dense = dense_bytes_per_device * P  # total across cluster
    active_per_device = []
    sparse_bytes = 0
    for d in range(P):
        count = int(jnp.sum(masks_per_device[d]))
        active_per_device.append(count)
        sparse_bytes += count * bytes_per_block
    return {
        'dense_bytes_total': int(total_dense),
        'sparse_bytes_total': int(sparse_bytes),
        'savings_bytes': int(total_dense - sparse_bytes),
        'savings_pct': (1 - sparse_bytes / total_dense) * 100 if total_dense > 0 else 0,
        'active_per_device': active_per_device,
        'blocks_per_device': local_blocks,
    }
# ============================================================
# Correctness Check
# ============================================================
print("=" * 70)
print("  GATE E.1: Correctness — Sharded vs Single-Device")
print("=" * 70)
seq = 16384
q, k, v, q_sh, k_sh, v_sh = make_sharded_data(seq)
local_blocks = (seq // P) // BS
# Dense reference
out_ref = dense_attn_single(q, k, v)
# Sharded dense (AllGather full KV)
out_sharded_dense = sharded_dense_attn(q_sh, k_sh, v_sh)
err_dense = jnp.max(jnp.abs(out_ref.astype(jnp.float32) - out_sharded_dense[0].astype(jnp.float32)))
print(f"  Dense single vs sharded-dense: max err = {err_dense:.6f}")
# Sharded OrthoCache (0% eviction — should match dense)
masks_0 = make_uniform_masks(local_blocks, 0)
out_ortho_0 = sharded_orthocache_attn(q_sh, k_sh, v_sh, masks_0)
err_ortho_0 = jnp.max(jnp.abs(out_ref.astype(jnp.float32) - out_ortho_0[0].astype(jnp.float32)))
print(f"  Dense single vs ortho-sharded (0% evict): max err = {err_ortho_0:.6f}")
# Sharded OrthoCache (50% eviction — matches predicated)
masks_50 = make_uniform_masks(local_blocks, 50)
out_ortho_50 = sharded_orthocache_attn(q_sh, k_sh, v_sh, masks_50)
# Compare against single-device predicated
@jax.jit
def predicated_single(q, k, v, masks_flat):
    sc = jnp.sqrt(jnp.float32(HD))
    full_mask = jnp.repeat(masks_flat, BS)
    lo = jnp.einsum('qhd,khd->qkh', q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    lo = jnp.where(full_mask[None, :, None], lo, -1e9)
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w, v.astype(jnp.float32)).astype(jnp.bfloat16)
# Reconstruct global mask from per-device masks
global_mask_50 = masks_50.reshape(-1)  # (P * local_blocks,)
out_pred_50 = predicated_single(q, k, v, global_mask_50)
err_ortho_50 = jnp.max(jnp.abs(out_pred_50.astype(jnp.float32) - out_ortho_50[0].astype(jnp.float32)))
print(f"  Predicated single vs ortho-sharded (50% evict): max err = {err_ortho_50:.6f}")
print()
# ============================================================
# GATE E.2: Data Volume Accounting
# ============================================================
print("=" * 70)
print("  GATE E.2: ICI Data Volume Accounting")
print("=" * 70)
for seq in [16384, 32768, 65536]:
    local_blocks = (seq // P) // BS
    print(f"\n  Seq={seq} ({seq//BS} blocks, {local_blocks}/device)")
    for pct in [0, 50, 75, 90]:
        masks = make_uniform_masks(local_blocks, pct)
        acct = ici_accounting(seq, masks)
        print(f"    {pct}% evict: dense={acct['dense_bytes_total']/1e6:.1f}MB "
              f"sparse={acct['sparse_bytes_total']/1e6:.1f}MB "
              f"saved={acct['savings_bytes']/1e6:.1f}MB ({acct['savings_pct']:.0f}%) "
              f"active/dev={acct['active_per_device']}")
# Skewed mask test
print(f"\n  Skewed masks (16K tokens):")
local_blocks = (16384 // P) // BS
masks_skew, rates = make_skewed_masks(local_blocks)
acct_skew = ici_accounting(16384, masks_skew)
print(f"    rates={rates}")
print(f"    active/dev={acct_skew['active_per_device']}")
print(f"    dense={acct_skew['dense_bytes_total']/1e6:.1f}MB "
      f"sparse={acct_skew['sparse_bytes_total']/1e6:.1f}MB "
      f"saved={acct_skew['savings_bytes']/1e6:.1f}MB ({acct_skew['savings_pct']:.0f}%)")
print()
# ============================================================
# GATE E.3: Latency Benchmark
# ============================================================
print("=" * 70)
print("  GATE E.3: Latency — Sharded Dense vs Sharded OrthoCache")
print("=" * 70)
results = {}
for seq in [8192, 16384, 32768]:
    local_blocks = (seq // P) // BS
    q_f, k_f, v_f, q_sh, k_sh, v_sh = make_sharded_data(seq)
    print(f"\n  {seq} tokens ({local_blocks} blocks/device)")
    t_dense = bench("sharded_dense",
                    lambda q=q_sh, k=k_sh, v=v_sh: sharded_dense_attn(q, k, v))
    for pct in [0, 50, 75, 90]:
        masks = make_uniform_masks(local_blocks, pct)
        # Warmup OrthoCache
        for _ in range(3):
            sharded_orthocache_attn(q_sh, k_sh, v_sh, masks).block_until_ready()
        t_ortho = bench(f"ortho_{pct}%",
                       lambda q=q_sh, k=k_sh, v=v_sh, m=masks:
                           sharded_orthocache_attn(q, k, v, m))
        dt = (t_dense - t_ortho) / t_dense * 100
        acct = ici_accounting(seq, masks)
        print(f"    Δτ = {dt:+.1f}%  ({t_dense/t_ortho:.2f}x)  "
              f"ICI saved: {acct['savings_pct']:.0f}%")
        results[(seq, pct)] = {
            'dense_ms': t_dense, 'ortho_ms': t_ortho,
            'dtau': dt, 'ici_saved_pct': acct['savings_pct'],
        }
# ============================================================
# Skewed mask latency test
# ============================================================
print(f"\n  Skewed masks (16K):")
q_f, k_f, v_f, q_sh, k_sh, v_sh = make_sharded_data(16384)
local_blocks = (16384 // P) // BS
masks_skew, rates = make_skewed_masks(local_blocks)
for _ in range(3):
    sharded_orthocache_attn(q_sh, k_sh, v_sh, masks_skew).block_until_ready()
t_skew = bench(f"ortho_skewed (rates={rates})",
               lambda: sharded_orthocache_attn(q_sh, k_sh, v_sh, masks_skew))
# ============================================================
# Summary
# ============================================================
print(f"\n{'='*70}")
print("  SUMMARY")
print(f"{'='*70}")
print(f"{'seq':>6} | {'ev%':>4} | {'dense':>7} | {'ortho':>7} | {'Δτ':>7} | {'ICI saved':>9}")
print("-" * 55)
for (seq, pct), r in sorted(results.items()):
    print(f"{seq:>6} | {pct:>4} | {r['dense_ms']:>6.3f} | {r['ortho_ms']:>6.3f} | "
          f"{r['dtau']:>6.1f}% | {r['ici_saved_pct']:>7.0f}%")

Platform: TPU v5 lite, JAX 0.10.1
Devices: 8 (TPU_0(process=0,(0,0,0,0)), TPU_1(process=0,(1,0,0,0)), TPU_2(process=0,(0,1,0,0)), TPU_3(process=0,(1,1,0,0)), TPU_4(process=0,(0,2,0,0)), TPU_5(process=0,(1,2,0,0)), TPU_6(process=0,(0,3,0,0)), TPU_7(process=0,(1,3,0,0)))
Config: Q=1, heads=4, d=128, block=512, devices=8
Method: pmap AllToAllv (Strategy C: static-buffer exchange)

  GATE E.1: Correctness — Sharded vs Single-Device
  Dense single vs sharded-dense: max err = 0.000000
  Dense single vs ortho-sharded (0% evict): max err = 0.000000
  Predicated single vs ortho-sharded (50% evict): max err = 0.000004

  GATE E.2: ICI Data Volume Accounting

  Seq=16384 (32 blocks, 4/device)
    0% evict: dense=16.8MB sparse=16.8MB saved=0.0MB (0%) active/dev=[4, 4, 4, 4, 4, 4, 4, 4]
    50% evict: dense=16.8MB sparse=8.4MB saved=8.4MB (50%) active/dev=[2, 2, 2, 2, 2, 2, 2, 2]
    75% evict: dense=16.8MB sparse=4.2MB saved=12.6MB (75%) active/dev=[1, 1, 1, 1, 1, 1, 1, 1]
    90% evict: dense=16.

In [2]:
"""Phase E.2: Stratified AllGather via shard_map — True ICI Bandwidth Reduction.
Platform: TPU v5e-8 (8 chips, 128 GB HBM), JAX 0.10.1
Method: shard_map + lax.switch over 4 pre-compiled AllGather capacity buckets
Architecture:
  1. Each device compacts local KV blocks (fancy-index reorder)
  2. pmax consensus → global_k_max (nanosecond scalar sync)
  3. lax.switch selects bucket (25%, 50%, 75%, 100% capacity)
  4. Selected branch: slice[:target_k] → all_gather → masked einsum
  5. Output: (1, NH, HD) replicated — softmax is complete on each device
Key difference from Strategy C (pmap):
  - Slice happens BEFORE the collective → physically fewer bytes on ICI
  - Dense einsum (not fori_loop) → MXU-parallel attention
  - lax.switch with static shapes → zero recompilation across masks
Paste this entire cell into a Kaggle TPU v5e notebook.
"""
import jax
import jax.numpy as jnp
from jax.experimental.shard_map import shard_map
from jax.sharding import Mesh, PartitionSpec as PS
from functools import partial
import time
# ============================================================
# Constants
# ============================================================
BS = 512    # Block size (tokens per block)
HD = 128    # Head dimension
NH = 4      # Attention heads (local per device)
WARMUP = 5
REPS = 20
print(f"Platform: {jax.devices()[0].device_kind}, JAX {jax.__version__}")
NDev = jax.device_count()
devices = jax.devices()[:NDev]
print(f"Devices: {NDev} ({', '.join(str(d) for d in devices)})")
print(f"Config: Q=1, heads={NH}, d={HD}, block={BS}, devices={NDev}")
print(f"Method: shard_map + Stratified AllGather (4 buckets)\n")
import numpy as np
mesh = Mesh(np.array(devices).reshape(-1), ('tp',))
# ============================================================
# Utilities
# ============================================================
def make_mask(nb, pct):
    """Create block eviction mask. True = active (kept)."""
    if pct == 0:
        return jnp.ones(nb, dtype=jnp.bool_)
    n_evict = max(1, int(nb * pct / 100))
    n_evict = min(n_evict, nb - 1)  # keep at least 1
    m = jnp.ones(nb, dtype=jnp.bool_).at[:n_evict].set(False)
    return m[jax.random.permutation(jax.random.PRNGKey(42), nb)]
def stream_compact(mask):
    """Sort active blocks to front. Returns (sorted_indices, num_active)."""
    nb = mask.shape[0]
    iota = jnp.arange(nb, dtype=jnp.int32)
    keys = jnp.where(mask, iota, nb + iota)
    return jnp.argsort(keys, stable=True), jnp.sum(mask).astype(jnp.int32)
def bench(label, fn):
    """Median-of-sorted latency benchmark."""
    for _ in range(WARMUP):
        fn().block_until_ready()
    ts = []
    for _ in range(REPS):
        t0 = time.perf_counter()
        fn().block_until_ready()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort()
    med = ts[len(ts) // 2]
    return med
def make_data(sl):
    """Generate random Q, K, V tensors."""
    k1, k2, k3 = jax.random.split(jax.random.PRNGKey(0), 3)
    q = jax.random.normal(k1, (1, NH, HD), dtype=jnp.bfloat16)
    k = jax.random.normal(k2, (sl, NH, HD), dtype=jnp.bfloat16)
    v = jax.random.normal(k3, (sl, NH, HD), dtype=jnp.bfloat16)
    return q, k, v
def prepare_inputs(nb_per_dev, pct):
    """Build global index/count arrays for shard_map sharding.
    Returns:
        global_indices: (NDev * nb_per_dev,) — sharded P('tp') → (nb_per_dev,) per device
        global_counts:  (NDev,)             — sharded P('tp') → (1,) per device
    """
    idx_list, cnt_list = [], []
    for d in range(NDev):
        mask = make_mask(nb_per_dev, pct)
        idx, n = stream_compact(mask)
        idx_list.append(idx)
        cnt_list.append(n.reshape(1))
    return jnp.concatenate(idx_list), jnp.concatenate(cnt_list)
# ============================================================
# Single-device references
# ============================================================
@jax.jit
def dense_single(q, k, v):
    """Full dense attention — gold reference."""
    sc = jnp.sqrt(jnp.float32(HD))
    lo = jnp.einsum('qhd,khd->qkh',
                     q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w,
                       v.astype(jnp.float32)).astype(jnp.bfloat16)
@jax.jit
def predicated_single(q, k, v, block_mask):
    """Masked attention — reference for eviction correctness."""
    sc = jnp.sqrt(jnp.float32(HD))
    tmask = jnp.repeat(block_mask, BS)
    lo = jnp.einsum('qhd,khd->qkh',
                     q.astype(jnp.float32), k.astype(jnp.float32)) / sc
    lo = jnp.where(tmask[None, :, None], lo, jnp.float32(-1e9))
    w = jax.nn.softmax(lo, axis=1)
    return jnp.einsum('qkh,khd->qhd', w,
                       v.astype(jnp.float32)).astype(jnp.bfloat16)
# ============================================================
# Sharded Dense Baseline (shard_map — full AllGather)
# ============================================================
def build_sharded_dense(mesh_):
    """Baseline: AllGather full K/V, dense einsum. No eviction."""
    @jax.jit
    @partial(shard_map, mesh=mesh_,
             in_specs=(PS(None, None, None),    # q: replicated (1, NH, HD)
                       PS('tp', None, None),     # k: seq-sharded (SL, NH, HD)
                       PS('tp', None, None)),    # v: seq-sharded
             out_specs=PS(None, None, None),      # output: replicated
             check_rep=False)
    def fn(q, k, v):
        k_all = jax.lax.all_gather(k, axis_name='tp', axis=0, tiled=True)
        v_all = jax.lax.all_gather(v, axis_name='tp', axis=0, tiled=True)
        sc = jnp.sqrt(jnp.float32(HD))
        lo = jnp.einsum('qhd,khd->qkh',
                         q.astype(jnp.float32),
                         k_all.astype(jnp.float32)) / sc
        w = jax.nn.softmax(lo, axis=1)
        return jnp.einsum('qkh,khd->qhd', w,
                           v_all.astype(jnp.float32)).astype(jnp.bfloat16)
    return fn
# ============================================================
# Stratified OrthoCache (shard_map + Bucketed AllGather)
# ============================================================
def build_stratified(mesh_, nb_per_dev):
    """Builds a JIT-compiled stratified AllGather attention kernel.
    Pre-compiles 4 discrete AllGather capacity profiles (b1..b4).
    At runtime, pmax consensus selects the tightest bucket that fits
    the global maximum active count. The selected branch:
      1. Slices K/V to target_k blocks (ICI savings)
      2. AllGathers the slice (physically smaller transfer)
      3. Builds validity mask from per-device counts
      4. Runs dense masked einsum (compute proportional to bucket)
    Args:
        mesh_: JAX Mesh object
        nb_per_dev: number of blocks per device (compile-time constant)
    Returns:
        JIT-compiled shard_map kernel
    """
    b1 = max(1, nb_per_dev // 4)
    b2 = max(1, nb_per_dev // 2)
    b3 = max(1, (nb_per_dev * 3) // 4)
    b4 = nb_per_dev
    # Deduplicate bucket values (at small nb, some may collide)
    buckets = []
    seen = set()
    for b in [b1, b2, b3, b4]:
        if b not in seen:
            buckets.append(b)
            seen.add(b)
    # Pad to exactly 4 entries (lax.switch needs fixed branch count)
    while len(buckets) < 4:
        buckets.append(buckets[-1])
    b1, b2, b3, b4 = buckets
    def make_branch(target_k):
        """Factory: creates one AllGather+Attention branch for a fixed capacity."""
        def branch(q, k_act, v_act, all_counts):
            # 1. Slice to bucket capacity (BEFORE the collective)
            k_sl = k_act[:target_k]   # (target_k, BS, NH, HD) — static shape
            v_sl = v_act[:target_k]
            # 2. AllGather: each device sends target_k blocks
            k_g = jax.lax.all_gather(k_sl, axis_name='tp', axis=0, tiled=True)
            v_g = jax.lax.all_gather(v_sl, axis_name='tp', axis=0, tiled=True)
            # Shape: (target_k * NDev, BS, NH, HD)
            # 3. Build block validity mask from per-device counts
            ntb = target_k * NDev
            bi = jnp.arange(ntb, dtype=jnp.int32)
            did = bi // target_k           # which device contributed this block
            lpos = bi % target_k           # local position within device's chunk
            sdid = jnp.minimum(did, NDev - 1)
            valid = (did < NDev) & (lpos < all_counts[sdid])
            tmask = jnp.repeat(valid, BS)  # token-level mask
            # 4. Dense masked einsum attention
            kf = k_g.reshape(-1, NH, HD)   # (ntb * BS, NH, HD)
            vf = v_g.reshape(-1, NH, HD)
            sc = jnp.sqrt(jnp.float32(HD))
            lo = jnp.einsum('qhd,khd->qkh',
                             q.astype(jnp.float32),
                             kf.astype(jnp.float32)) / sc
            lo = jnp.where(tmask[None, :, None], lo, jnp.float32(-1e9))
            w = jax.nn.softmax(lo, axis=1)
            return jnp.einsum('qkh,khd->qhd', w,
                               vf.astype(jnp.float32)).astype(jnp.bfloat16)
        return branch
    @jax.jit
    @partial(shard_map, mesh=mesh_,
             in_specs=(PS(None, None, None),    # q: replicated
                       PS('tp', None, None),     # k: seq-sharded
                       PS('tp', None, None),     # v: seq-sharded
                       PS('tp'),                 # active_indices: per-device (nb_per_dev,)
                       PS('tp')),                # num_active: per-device (1,)
             out_specs=PS(None, None, None),      # output: replicated
             check_rep=False)
    def kernel(q, k, v, active_indices, num_active):
        nb = k.shape[0] // BS
        # Reshape to block layout
        kb = k.reshape(nb, BS, NH, HD)
        vb = v.reshape(nb, BS, NH, HD)
        # Reindex: active blocks to front via fancy indexing (XLA gather)
        ka = kb[active_indices]
        va = vb[active_indices]
        # Scalar consensus: global maximum active count
        na = num_active[0]  # squeeze (1,) → scalar
        gmax = jax.lax.pmax(na, axis_name='tp')
        # AllGather per-device counts for mask building
        ac = jax.lax.all_gather(num_active, axis_name='tp',
                                axis=0, tiled=True)  # (NDev,)
        # Bucket selection (same on all devices → same branch)
        bidx = jnp.where(gmax <= b1, 0,
               jnp.where(gmax <= b2, 1,
               jnp.where(gmax <= b3, 2, 3)))
        # Dispatch to pre-compiled branch
        return jax.lax.switch(bidx, [
            make_branch(b1),
            make_branch(b2),
            make_branch(b3),
            make_branch(b4),
        ], q, ka, va, ac)
    return kernel, (b1, b2, b3, b4)
# ════════════════════════════════════════════════════════════════
#   GATE E.1b: Correctness
# ════════════════════════════════════════════════════════════════
print("=" * 70)
print("  GATE E.1b: Correctness — shard_map Stratified AllGather")
print("=" * 70)
seq_c = 16384
q_c, k_c, v_c = make_data(seq_c)
nb_c = seq_c // (BS * NDev)
# Dense reference
out_ref = dense_single(q_c, k_c, v_c)
# Sharded dense baseline
sd_fn = build_sharded_dense(mesh)
out_sd = sd_fn(q_c, k_c, v_c)
err_sd = float(jnp.max(jnp.abs(out_ref.astype(jnp.float32) -
                                 out_sd.astype(jnp.float32))))
print(f"  Dense single vs shard_map dense: max err = {err_sd:.6f}")
# Stratified 0% eviction (should match dense exactly)
strat_fn, bkts = build_stratified(mesh, nb_c)
idx_0, cnt_0 = prepare_inputs(nb_c, 0)
out_s0 = strat_fn(q_c, k_c, v_c, idx_0, cnt_0)
err_s0 = float(jnp.max(jnp.abs(out_ref.astype(jnp.float32) -
                                 out_s0.astype(jnp.float32))))
print(f"  Dense single vs stratified (0% evict): max err = {err_s0:.6f}")
# Stratified 50% eviction (should match predicated reference)
idx_50, cnt_50 = prepare_inputs(nb_c, 50)
out_s50 = strat_fn(q_c, k_c, v_c, idx_50, cnt_50)
# Build global mask for predicated reference (same mask repeated per device)
gmask_50 = jnp.concatenate([make_mask(nb_c, 50) for _ in range(NDev)])
out_pred = predicated_single(q_c, k_c, v_c, gmask_50)
err_s50 = float(jnp.max(jnp.abs(out_pred.astype(jnp.float32) -
                                  out_s50.astype(jnp.float32))))
print(f"  Predicated single vs stratified (50% evict): max err = {err_s50:.6f}")
print()
# ════════════════════════════════════════════════════════════════
#   GATE E.2b: ICI Data Volume Accounting
# ════════════════════════════════════════════════════════════════
print("=" * 70)
print("  GATE E.2b: ICI Data Volume — Stratified Buckets vs Dense")
print("=" * 70)
for seq_v in [16384, 32768, 65536]:
    nb_v = seq_v // (BS * NDev)
    _, bk_v = build_stratified(mesh, nb_v)
    bv1, bv2, bv3, bv4 = bk_v
    # Dense: each device sends nb_v blocks via AllGather
    # Total ICI per device = nb_v * BS * NH * HD * dtype_bytes
    bytes_per_block = BS * NH * HD * 2  # bf16
    dense_ici = nb_v * bytes_per_block  # per-device send volume
    print(f"\n  Seq={seq_v} ({seq_v // BS} blocks, {nb_v}/dev, "
          f"buckets=[{bv1},{bv2},{bv3},{bv4}])")
    for pct in [0, 50, 75, 90]:
        mask = make_mask(nb_v, pct)
        _, na = stream_compact(mask)
        na = int(na)
        # Bucket selection logic (mirrors kernel)
        if na <= bv1:   bucket, bn = bv1, f"b1={bv1}"
        elif na <= bv2: bucket, bn = bv2, f"b2={bv2}"
        elif na <= bv3: bucket, bn = bv3, f"b3={bv3}"
        else:           bucket, bn = bv4, f"b4={bv4}"
        strat_ici = bucket * bytes_per_block
        saved_pct = (1 - strat_ici / dense_ici) * 100 if dense_ici > 0 else 0
        print(f"    {pct:>2}% evict: K={na} → {bn}  "
              f"dense={dense_ici * NDev / 1e6:.1f}MB  "
              f"strat={strat_ici * NDev / 1e6:.1f}MB  "
              f"ICI saved={saved_pct:.0f}%")
print()
# ════════════════════════════════════════════════════════════════
#   GATE E.3b: Latency Benchmark
# ════════════════════════════════════════════════════════════════
print("=" * 70)
print("  GATE E.3b: Latency — shard_map Dense vs Stratified OrthoCache")
print("=" * 70)
results = {}
for seq_b in [8192, 16384, 32768, 65536]:
    nb_b = seq_b // (BS * NDev)
    if nb_b < 1:
        print(f"\n  {seq_b} tokens: skipped (< 1 block/dev)")
        continue
    q_b, k_b, v_b = make_data(seq_b)
    d_fn = build_sharded_dense(mesh)
    o_fn, bk_b = build_stratified(mesh, nb_b)
    print(f"\n  {seq_b} tokens ({nb_b} blocks/dev, "
          f"buckets=[{bk_b[0]},{bk_b[1]},{bk_b[2]},{bk_b[3]}])")
    t_dense = bench("shard_dense", lambda: d_fn(q_b, k_b, v_b))
    print(f"  shard_dense: {t_dense:.3f} ms")
    for pct in [0, 50, 75, 90]:
        idx_b, cnt_b = prepare_inputs(nb_b, pct)
        # Capture loop vars
        t_o = bench(f"strat_{pct}%",
                    lambda _i=idx_b, _c=cnt_b: o_fn(q_b, k_b, v_b, _i, _c))
        mask_b = make_mask(nb_b, pct)
        _, na_b = stream_compact(mask_b)
        na_b = int(na_b)
        # Determine bucket
        if na_b <= bk_b[0]:   bkt = bk_b[0]
        elif na_b <= bk_b[1]: bkt = bk_b[1]
        elif na_b <= bk_b[2]: bkt = bk_b[2]
        else:                 bkt = bk_b[3]
        ici_pct = (1 - bkt / nb_b) * 100
        dt = (t_dense - t_o) / t_dense * 100
        spd = t_dense / t_o
        print(f"  strat_{pct}% (K={na_b}, bucket={bkt}): {t_o:.3f} ms  "
              f"Δτ = {dt:+.1f}%  ({spd:.2f}x)  ICI saved={ici_pct:.0f}%")
        results[(seq_b, pct)] = {
            'dense': t_dense, 'ortho': t_o, 'dtau': dt,
            'ici_saved': ici_pct, 'K': na_b, 'bucket': bkt
        }
# ════════════════════════════════════════════════════════════════
#   Summary Table
# ════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  SUMMARY: shard_map Stratified AllGather")
print(f"{'=' * 70}")
print(f"{'seq':>6} | {'ev%':>4} | {'K':>3} | {'bkt':>3} | "
      f"{'dense':>8} | {'strat':>8} | {'Δτ':>8} | {'ICI↓':>5}")
print("-" * 65)
for (seq_s, pct_s), r in sorted(results.items()):
    print(f"{seq_s:>6} | {pct_s:>4} | {r['K']:>3} | {r['bucket']:>3} | "
          f"{r['dense']:>7.3f} | {r['ortho']:>7.3f} | "
          f"{r['dtau']:>+7.1f}% | {r['ici_saved']:>4.0f}%")

Platform: TPU v5 lite, JAX 0.10.1
Devices: 8 (TPU_0(process=0,(0,0,0,0)), TPU_1(process=0,(1,0,0,0)), TPU_2(process=0,(0,1,0,0)), TPU_3(process=0,(1,1,0,0)), TPU_4(process=0,(0,2,0,0)), TPU_5(process=0,(1,2,0,0)), TPU_6(process=0,(0,3,0,0)), TPU_7(process=0,(1,3,0,0)))
Config: Q=1, heads=4, d=128, block=512, devices=8
Method: shard_map + Stratified AllGather (4 buckets)

  GATE E.1b: Correctness — shard_map Stratified AllGather
  Dense single vs shard_map dense: max err = 0.000000
  Dense single vs stratified (0% evict): max err = 0.000000
  Predicated single vs stratified (50% evict): max err = 0.000000

  GATE E.2b: ICI Data Volume — Stratified Buckets vs Dense

  Seq=16384 (32 blocks, 4/dev, buckets=[1,2,3,4])
     0% evict: K=4 → b4=4  dense=16.8MB  strat=16.8MB  ICI saved=0%
    50% evict: K=2 → b2=2  dense=16.8MB  strat=8.4MB  ICI saved=50%
    75% evict: K=1 → b1=1  dense=16.8MB  strat=4.2MB  ICI saved=75%
    90% evict: K=1 → b1=1  dense=16.8MB  strat=4.2MB  ICI saved=75%

  Se